# 05 - Baselines

Logistic regression, random forest, SVM. Required for the ablation table, not the contribution. Every run writes per-domain predictions so metrics can be recomputed later without retraining.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'

if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
else:
    # Private repo: paste your GitHub Personal Access Token when prompted.
    # It is only held in this runtime and vanishes when the session ends.
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{URL}', REPO], check=True)

sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))


In [ ]:
import pandas as pd
from src.models import baselines
from src.evaluate import splits, metrics, predictions
from src.features.build import to_matrix

cfg = config.load('baseline')
df = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet")
split = splits.load_split(P['data']['splits'], cfg['split']['name'])
tr, va, te = splits.apply_split(df, split)

In [ ]:
for name, params in cfg['models'].items():
    Xtr, ytr, _ = to_matrix(tr); Xte, yte, _ = to_matrix(te)
    model = baselines.build(name, params).fit(Xtr.fillna(-1), ytr)
    scores = model.predict_proba(Xte.fillna(-1))[:, 1]
    m = metrics.evaluate(yte, scores)

    counter = manifest.next_counter(P['manifest'])
    run_id = manifest.make_run_id(name, cfg['split']['name'], cfg['seed'], counter)
    predictions.save(run_id, P['artifacts']['predictions'], te['domain'], yte, scores)
    manifest.record(P['manifest'], run_id, 'baseline', cfg, cfg['split']['name'],
                    split['split_file'], m, cfg['seed'], repo_root=REPO)
    print(run_id, {k: round(v,4) for k,v in m.items() if isinstance(v,float)})